---
### Follow the Workers: v2 analysis (diagnostics, v2 variants, forward test)
---

---
##### Imports
Project modules; the frozen study is only ever imported read-only inside data.py.

In [1]:
import hashlib
import json
import warnings
from datetime import datetime, timezone

import numpy as np
import pandas as pd

import params as P
import data as D
import stats as S
import plots as G

# numpy 2.0 with Apple Accelerate raises spurious "encountered in matmul" warnings; Gate 1 below
# shows results match the frozen study to ~1e-16, so they are silenced (only that message).
warnings.filterwarnings('ignore', message='.*encountered in matmul', category=RuntimeWarning)
pd.set_option('display.width', 160)
for folder in [P.OUT, P.TABLES, P.FIGS, P.TEX, P.FORWARD]:
    folder.mkdir(parents=True, exist_ok=True)


def utc_now():
    return datetime.now(timezone.utc).isoformat(timespec='seconds')


def sha256(path):
    return hashlib.sha256(open(path, 'rb').read()).hexdigest()


def stop(reason):
    """A failed gate: record it and halt the notebook (CLAUDE.md rule 6)."""
    (P.OUT / 'STOP.md').write_text(f'# STOP\n\n{utc_now()}\n\n{reason}\n')
    raise RuntimeError(reason)


def frozen_fingerprint():
    files = sorted(p for p in P.FROZEN_REPO.rglob('*') if p.is_file() and '.git' not in p.relative_to(P.FROZEN_REPO).parts)
    h = hashlib.sha256()
    for p in files:
        h.update(str(p.relative_to(P.FROZEN_REPO)).encode())
        h.update(hashlib.sha256(p.read_bytes()).digest())
    return h.hexdigest(), len(files)

---
##### Data
Public French returns, current FRED labor data with fixed publication lags, and the frozen study's saved outputs.

In [2]:
R = D.load_returns()                 # 13 group returns, excess, relative, factors
F, T = D.signals()                   # standardised F1..F5, W and tightness T, by earning month
F['A0'] = (F['F1'] + F['F2'] - F['F4'] + F['F5']) / 4      # preregistered fixed rule
F['IndMom'] = D.industry_momentum(R['total'])             # industry momentum 12-1 (months m-12..m-2)
FP, FD = D.frozen()                  # frozen params.py and data.py (read-only)
print('returns', R['total'].index.min(), '->', R['total'].index.max(),
      '| signals', F['W'].dropna(how='all').index.min(), '->', F['W'].dropna(how='all').index.max())

returns 1963-07 -> 2026-08 | signals 2004-04 -> 2026-11


In [3]:
# Which data vintage is used (ITERATION_PROMPT Part 1 step 2), with the evidence for the choice.
panels_in_repo = sorted(str(p.relative_to(P.FROZEN_REPO)) for p in P.FROZEN_REPO.rglob('*.parquet'))
env_key = (P.ROOT / '.env').exists() and 'FRED_API_KEY' in (P.ROOT / '.env').read_text()
vintage_check = {}
for variant in ['FIXEDLAG', 'ASOF', 'REVISED', 'FIRST']:
    saved = D.saved_csv(f'{variant}_A0_scores.csv')
    ours = F['A0'].reindex(saved.index)
    vintage_check[variant] = {
        'pooled_corr': float(np.corrcoef(ours.to_numpy().ravel(), saved.to_numpy().ravel())[0, 1]),
        'mean_monthly_rank_corr': float(S.rank_ic(ours, saved).mean()),
        'same_top3_share': float(np.mean([set(ours.loc[m].nlargest(3).index) == set(saved.loc[m].nlargest(3).index) for m in saved.index]))}
vintage = pd.DataFrame(vintage_check).T
vintage.index.name = 'saved A0 variant'
res = json.loads((P.SEALED / 'results.json').read_text())
ic_a3 = {v: res['variants'][v]['A3']['IC']['1']['mean'] for v in ['ASOF', 'FIXEDLAG']}
choice = 'a' if panels_in_repo else 'b' if env_key else 'c'
text = f"""# Data vintage used in this iteration

Written by analysis/run.ipynb on {utc_now()}.

**Choice: option ({choice})** of ITERATION_PROMPT Part 1 step 2.

- (a) As-known (ASOF) panels saved in the frozen repo: **not available**. The repo contains {len(panels_in_repo)} parquet files; its package_manifest.json lists the panels as omitted.
- (b) ALFRED vintages with `analysis/.env` FRED_API_KEY: **not available** (no key file).
- (c) Revised FRED data downloaded 3 Oct 2026 from fredgraph.csv, with fixed publication lags (JOLTS month d-2, CES month d-1, tightness month d-2): **used** for every historical result in this iteration.

The forward-test book (Part 3) does not use (c). It uses ALFRED values *as published on {P.VINTAGE_DATE}*, downloaded without a key from alfredgraph.csv, so it only contains data known before the 30 Sep 2026 decision.

## How close is (c) to the frozen study's own signals?

Our rebuilt A0 composite (same transforms) against the frozen study's saved sealed A0 scores, Nov 2014 to Aug 2026:

{G.markdown(vintage.round(3))}

Our data is the FIXEDLAG construction (revised values, fixed lags), and it reproduces the frozen FIXEDLAG scores almost exactly. As-known (ASOF) data picks the same top three industries in only {vintage_check['ASOF']['same_top3_share']:.0%} of months, so revisions do change portfolio membership. The frozen study's primary A3 had the same test IC under ASOF ({ic_a3['ASOF']:+.4f}) and FIXEDLAG ({ic_a3['FIXEDLAG']:+.4f}) (results.json), so the conclusions are not sensitive to this choice, but monthly holdings are. Every historical number in this iteration is therefore labelled as computed on revised data.
"""
(P.OUT / 'data_vintage.md').write_text(text)
vintage.round(4)

,pooled_corr,mean_monthly_rank_corr,same_top3_share
saved A0 variant,,,
FIXEDLAG,0.9983,0.9922,0.9085
ASOF,0.9181,0.8635,0.4155
REVISED,0.9912,0.9793,0.8592
FIRST,0.8506,0.7796,0.2394


---
##### Gate 1
Reproduce the sealed A0 net returns from its saved scores with the frozen backtest, and the 13 group returns against every saved ledger.

In [4]:
checks = []

# 1. The frozen repo is unchanged since the baseline fingerprint.
fingerprint, n_files = frozen_fingerprint()
checks.append(('Frozen repo unchanged (SHA-256 over all files)', f'{n_files} files, {fingerprint[:12]}...',
               f'{P.FROZEN_FINGERPRINT[1]} files, {P.FROZEN_FINGERPRINT[0][:12]}...', (fingerprint, n_files) == P.FROZEN_FINGERPRINT))

# 2. Portfolio parameters equal the frozen study's.
same = all(FP.PORTFOLIO[k] == v for k, v in P.PORTFOLIO.items())
checks.append(('Portfolio parameters equal frozen params.PORTFOLIO', str(P.PORTFOLIO), 'identical', same))

# 3. Sealed A0 net monthly returns from the saved A0 scores, through the frozen backtest.
saved_ledger = D.saved_csv('ASOF_A0_ledger.csv')
ledger, _ = D.backtest(D.saved_csv('ASOF_A0_scores.csv'), R)
for col in ['net', 'gross', 'cost', 'turnover']:
    diff = float((ledger[col] - saved_ledger[col]).abs().max())
    checks.append((f'Sealed A0 {col}: max |ours - saved| over {len(ledger)} months', f'{diff:.2e}', '< 1e-6', diff < 1e-6))

# 4. Group returns: saved weights x our group excess returns + hedge = saved gross, every saved ledger.
worst, count = 0.0, 0
for study in [P.SONNET, P.OPUS]:
    sealed = sorted((study / 'outputs/sealed/initial').glob('*_weights.csv'))
    research = sorted((study / 'outputs').glob('research_A*_weights.csv'))
    for path in sealed + research:
        folder = 'outputs/sealed/initial' if path in sealed else 'outputs'
        w = D.saved_csv(path.name, study, folder)
        g = D.saved_csv(path.name.replace('_weights', '_ledger'), study, folder)
        groups = [c for c in w.columns if c != 'market']
        recon = (w[groups] * R['excess'].loc[w.index, groups]).sum(axis=1) + w['market'] * R['factors'].loc[w.index, 'Mkt-RF']
        worst = max(worst, float((recon - g.gross).abs().max()))
        count += 1
checks.append((f'13 group returns vs {count} saved ledgers (both studies, sealed + research)', f'{worst:.2e}', '< 1e-6', worst < 1e-6))

# 5. Our estimators reproduce the frozen results.json (IC, HAC t, Sharpe, factor regression).
res = json.loads((P.SEALED / 'results.json').read_text())
worst_est = 0.0
for s in ['A0', 'A1', 'A1-T', 'A3', 'A4']:
    l, sc = D.saved_csv(f'ASOF_{s}_ledger.csv'), D.saved_csv(f'ASOF_{s}_scores.csv')
    ic = S.rank_ic(sc, R['relative'])
    fa = S.factor_attribution(l.net, R)
    ref, ref_a = res['variants']['ASOF'][s], res['attribution'][s]
    diffs = [float((ic - D.saved_csv(f'ASOF_{s}_ic.csv')['ic']).abs().max()),
             abs(S.hac(ic, lags=6)['t'] - ref['IC']['1']['t']), abs(S.perf(l)['sharpe'] - ref['sharpe']),
             max(abs(fa['coef'][k] - ref_a['coefficients'][k]) for k in ref_a['coefficients']),
             max(abs(fa['tstat'][k] - ref_a['t_statistics'][k]) for k in ref_a['t_statistics'])]
    worst_est = max(worst_est, max(diffs))
checks.append(('Estimators vs results.json (IC, IC t, Sharpe, factor coefs and t; A0-A4)', f'{worst_est:.2e}', '< 1e-8', worst_est < 1e-8))

gate1 = pd.DataFrame(checks, columns=['check', 'value', 'threshold', 'pass'])
gate1

,check,value,threshold,pass
0,Frozen repo unchanged (SHA-256 over all files),"3067 files, eb109be96917...","3067 files, eb109be96917...",True
1,Portfolio parameters equal frozen params.PORTF...,"{'longs': 3, 'shorts': 3, 'holding': 3, 'risk_...",identical,True
2,Sealed A0 net: max |ours - saved| over 142 months,9.97e-17,< 1e-6,True
3,Sealed A0 gross: max |ours - saved| over 142 m...,9.71e-17,< 1e-6,True
4,Sealed A0 cost: max |ours - saved| over 142 mo...,9.94e-17,< 1e-6,True
5,Sealed A0 turnover: max |ours - saved| over 14...,2.22e-16,< 1e-6,True
6,13 group returns vs 50 saved ledgers (both stu...,1.51e-16,< 1e-6,True
7,"Estimators vs results.json (IC, IC t, Sharpe, ...",3.55e-15,< 1e-8,True


In [5]:
passed = bool(gate1['pass'].all())
lines = [f'# Gate 1 validation\n\nRun {utc_now()} by analysis/run.ipynb. **Result: {"PASS" if passed else "FAIL"}**\n',
         'Every check uses the frozen study\'s saved files in `230ga-follow-the-workers/` (read-only) and public Ken French data '
         '(CRSP 202608 build) in `analysis/data/french/`. The backtest is the frozen `data.backtest`, imported read-only.\n',
         G.markdown(gate1, index=False), '',
         '## Informational: rebuilt labor signals vs the frozen study\'s saved A0 scores', '',
         'Not part of the gate (our labor data is revised FRED, not the frozen as-known panels). See `data_vintage.md`.', '',
         G.markdown(vintage.round(4)), '']
(P.OUT / 'validation.md').write_text('\n'.join(lines))
if not passed:
    stop('Gate 1 failed. See analysis/output/validation.md:\n\n' + G.markdown(gate1, index=False))
print('Gate 1:', 'PASS' if passed else 'FAIL')

Gate 1: PASS


---
##### Portfolio equation
The frozen score-to-weight mapping, written as LaTeX for the report.

In [6]:
equation = r"""% Score-to-weight mapping of the frozen study's backtest (230ga-follow-the-workers/outputs/follow_the_workers/data.py::backtest).
% Written by analysis/run.ipynb. Parameters: analysis/params.py PORTFOLIO (checked equal to the frozen params in Gate 1).
\begin{align}
\tau_{m,g} &= \begin{cases} +\tfrac13 & g \in \text{top 3 of } s_{m,\cdot}\\ -\tfrac13 & g \in \text{bottom 3 of } s_{m,\cdot}\\ 0 & \text{otherwise}\end{cases}
  \qquad b_{m} = \frac{1}{H}\sum_{j=0}^{H-1} \tau_{m-j}, \quad H=3, \label{eq:tranche}\\
h_{m} &= -\sum_{g} b_{m,g}\,\hat\beta_{m,g}, \qquad x_m = (b_{m,1},\dots,b_{m,13},\,h_m), \label{eq:hedge}\\
k_m &= \min\!\left(\frac{0.10}{\sqrt{12\,x_m^{\top}\hat\Sigma_m x_m}},\ \frac{2}{\lVert x_m\rVert_1}\right), \qquad w_m = k_m\,x_m, \label{eq:scale}\\
r^{\mathrm{net}}_{m} &= \sum_{g} w_{m,g}\, r^{e}_{m,g} + w_{m,M}\,\mathrm{MKT}_m
  - c\sum_{g}\lvert w_{m,g}-\tilde w_{m,g}\rvert - c_h\,\lvert w_{m,M}-\tilde w_{m,M}\rvert. \label{eq:net}
\end{align}
where $s_{m,g}$ is the score of group $g$ known at the end of month $m-1$, $\hat\beta_{m,g}$ and the Ledoit--Wolf covariance
$\hat\Sigma_m$ (13 groups plus the market) use the 60 months ending in $m-2$ (at least 36), $r^{e}_{m,g}$ is the group's excess
return, $\tilde w$ are the previous weights after drift, $c=10$\,bp and $c_h=2$\,bp."""
(P.TEX / 'portfolio_equation.tex').write_text(equation + '\n')
print((P.TEX / 'portfolio_equation.tex').read_text()[:200])

% Score-to-weight mapping of the frozen study's backtest (230ga-follow-the-workers/outputs/follow_the_workers/data.py::backtest).
% Written by analysis/run.ipynb. Parameters: analysis/params.py PORTFO
